# 实验流程

本项目围绕 **LLM 数学推理评估（GSM8K Best-of-N & ProcessBench）** 展开，采用冻结编码器与轻量级奖励模型相结合的方法，对数学推理轨迹进行评分与重排序，并从数据生成、模型训练、外部评估、内在评估等多个角度分析奖励模型的性能。

整个实验流程如下：

```
Math-Shepherd
      │
      ▼
数据预处理
      │
      ▼
冻结 Qwen2.5-0.5B 提取特征
(precompute_embeddings.py)
      │
      ▼
Embedding Cache
      │
      ▼
训练 Reward Head
(train_from_cache.py)
      │
      ▼
 ┌──────────────┬──────────────┬──────────────┐
 ▼              ▼              ▼
Single Eval   BoN Eval    Benchmark
```

---

# 第一阶段：数据准备

实验主要涉及两类数据集。

## 1. Math-Shepherd

用于训练 Process Reward Model（PRM）。

主要文件：

```
data/
├── train.jsonl
├── val.jsonl
└── math_shepherd_full.jsonl
```

数据格式：

```json
{
    "question":"...",
    "steps":[...],
    "labels":[1,1,0,...]
}
```

其中：

- question：题目
- steps：推理步骤
- labels：每一步是否正确

---

## 2. GSM8K + ProcessBench

用于生成和评估 Best-of-N。

主要文件：

```
data/
├── gsm8k_qwen0.5b_bon16.jsonl
└── processbench_bon_gsm8k.jsonl
```

---

# 第二阶段：Embedding 预计算

由于编码器保持冻结，因此整个实验只需运行一次。

对应文件：

```
encoder.py
precompute_embeddings.py
```

训练集：

```bash
python precompute_embeddings.py \
    --train_file data/train.jsonl \
    --cache_dir cache/train_clean
```

验证集：

```bash
python precompute_embeddings.py \
    --train_file data/val.jsonl \
    --cache_dir cache/val_clean
```

输出：

```
cache/

train_clean/
val_clean/
```

之后所有训练均直接读取缓存，不再调用 LLM。

---

# 第三阶段：训练 Reward Head

对应文件：

```
reward_heads.py
pqm_loss.py
train_from_cache.py
```

目前实验包含四种奖励模型：

- Linear
- MLP
- CNN
- GRU

训练示例：

```bash
python train_from_cache.py \
    --cache_dir cache/train_clean \
    --head gru \
    --epochs 10 \
    --save_path checkpoints/gru_clean.pt
```

模型保存在：

```
checkpoints/

gru_clean.pt
cnn_clean.pt
mlp_clean.pt
linear_full.pt
```

同时生成：

```
results/

gru_efficiency.json
cnn_efficiency.json
...
```

记录：

- 参数量
- 峰值显存
- 训练耗时

---

# 第四阶段：Single Trajectory Evaluation

用于评估奖励模型对整条推理轨迹正确性的判别能力。

对应文件：

```
eval_single.py
```

运行：

```bash
python eval_single.py \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

输出：

```
results/

gru_single_metrics.json
```

主要指标：

- Pairwise Separation
- Accuracy

---

# 第五阶段：Oracle Evaluation

Oracle 用于计算生成模型理论性能上限。

对应文件：

```
eval_oracle.py
```

运行：

```bash
python eval_oracle.py \
    --eval_file data/gsm8k_qwen0.5b_bon16.jsonl
```

实验结果：

| 指标 | 数值 |
|------|------|
| Oracle@16 | **76.04%** |

说明：

若奖励模型能够始终选中正确轨迹，则理论最高正确率可达到 **76.04%**。

---

# 第六阶段：Best-of-N 外部评估

对应文件：

```
eval_bon.py
```

运行：

```bash
python eval_bon.py \
    --eval_file data/gsm8k_qwen0.5b_bon16.jsonl \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt \
    --ks 16
```

同时测试四种聚合策略：

- last
- min
- sum
- mean

实验结果：

| 聚合策略 | BoN@16 |
|----------|---------|
| last | **34.04%** |
| min | 32.37% |
| sum | 32.22% |
| mean | 31.99% |

---

# 第七阶段：ProcessBench 内在评估

对应文件：

```
eval_intrinsic.py
```

运行：

```bash
python eval_intrinsic.py \
    --eval_file data/processbench_bon_gsm8k.jsonl \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

实验结果：

| 指标 | 数值 |
|------|------|
| ROC-AUC | **0.6139** |
| Accuracy | **0.5000** |

说明：

奖励模型虽然具有一定排序能力，但整体判别性能仍然较弱。

---

# 第八阶段：效率测试

对应文件：

```
benchmark.py
```

运行：

```bash
python benchmark.py \
    --model_name Qwen/Qwen2.5-0.5B \
    --head gru \
    --batch_size 48 \
    --seq_len 512
```

统计：

- GPU 显存
- 吞吐率
- 每 Epoch 时间

---

# 第九阶段：结果汇总

对应文件：

```
summarize_results.py
```

运行：

```bash
python summarize_results.py
```

自动汇总：

```
results/

gru_single_metrics.json
gru_efficiency.json
cnn_single_metrics.json
...
```

生成最终实验统计结果。

---

# 项目文件对应关系

| 文件 | 功能 |
|------|------|
| `encoder.py` | 冻结 Qwen 编码器 |
| `reward_heads.py` | Linear、MLP、CNN、GRU 奖励头 |
| `pqm_loss.py` | PQM Ranking Loss |
| `dataset.py` | 数据读取 |
| `precompute_embeddings.py` | 预计算 Embedding |
| `train_from_cache.py` | 训练奖励模型 |
| `benchmark.py` | 效率测试 |
| `generate_bon_data.py` | 生成 GSM8K Best-of-N 数据 |
| `eval_single.py` | 单条轨迹评估 |
| `eval_oracle.py` | Oracle 理论上限 |
| `eval_bon.py` | Best-of-N 重排序评估 |
| `eval_intrinsic.py` | ProcessBench 内在评估 |
| `summarize_results.py` | 汇总实验结果 |

In [1]:
from datasets import load_dataset
import random
import json

# 抽取数量（100~200之间任选）
N = 150

# 固定随机种子，保证结果可复现
SEED = 42

# 加载训练集
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 随机抽样
random.seed(SEED)
indices = random.sample(range(len(dataset)), N)

# 保存为 JSONL
with open("dummy_train.jsonl", "w", encoding="utf-8") as f:
    for idx in indices:
        json.dump(dataset[idx], f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {N} samples to dummy_train.jsonl")

f:\anaconda3\envs\node2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 150 samples to dummy_train.jsonl


In [2]:
from datasets import load_dataset
import json

# 加载 Math-Shepherd 数据集（请替换为具体的 HF 仓库名，如 "math-shepherd/..."）
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 导出为你代码需要的 jsonl 格式
output_file = "data/math_shepherd_full.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
print(f"下载完成！共 {len(dataset)} 条数据，已保存至 {output_file}")

KeyboardInterrupt: 